# Blend: Ridge + weighted-KDE

**Date:** 2026-04-18.
**Hypothesis:** Ridge and weighted-KDE have complementary strengths. Their blend might dominate both.

|                  | cohort MAE | cohort me | h/m MAE | h/m me |
|---               |---         |---        |---      |---     |
| Weighted-KDE     | 13.26      | +1.90     | 18.23   | −6.78  |
| Ridge(α=10)      | **8.85**   | **−0.30** | 26.91   | **+0.69** |

**Blend candidates:**
1. Constant w: `pred = w*ridge + (1-w)*wkde` for w ∈ {0, 0.25, 0.5, 0.75, 1.0}.
2. Adaptive w based on `n_observed`: Ridge leans on target-local signals (rate_last_day dominates), so at high n_obs Ridge has more info. KDE relies more on cohort structure, helpful at low n_obs.
   - `w_ridge = min(1, n_obs/threshold)` for thresholds {40, 80, 120, 160}.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    critic_activity_counts, observed_review_stats,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SNAP_DAYS = 3

reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

activity = critic_activity_counts()
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'blend.pkl'
print('Ready.')

## Compute both predictions for every target

In [ ]:
FEATURES = ['observed_count','first_review_dbc','target_gap','observed_rate',
            'rate_last_day','rate_first_day','top_critic_frac','pub_diversity','pub_entropy','low_activity_frac']

def target_data(slug):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    close_midnight = target_close.floor('D')

    mr_all = reviews_noon[reviews_noon['movie_slug']==slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time) & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None
    first_review_ts_target = obs['estimated_timestamp'].min()
    first_review_dbc = (target_close - first_review_ts_target).total_seconds() / 86400
    obs_window_days = first_review_dbc - snap_dbc_eff
    if obs_window_days <= 0:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    stats = observed_review_stats(slug, first_review_ts_target, obs_window_days, activity)
    last_day_start = snap_time - pd.Timedelta(days=1)
    rate_last_day = ((obs['estimated_timestamp'] >= last_day_start) & (obs['estimated_timestamp'] < snap_time)).sum()
    first_day_end = first_review_ts_target + pd.Timedelta(days=1)
    rate_first_day = ((obs['estimated_timestamp'] >= first_review_ts_target) & (obs['estimated_timestamp'] < first_day_end)).sum()
    actual = int(((mr_all['estimated_timestamp'] >= snap_time) & (mr_all['estimated_timestamp'] < close_midnight)).sum())
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': first_review_dbc,
    }
    return {
        'slug': slug, 'snap_time': snap_time, 'snap_dbc_eff': snap_dbc_eff,
        'midnight_utc_dbc': midnight_utc_dbc, 'state': state,
        'target_gap': target_gap, 'obs_window_days': obs_window_days,
        'observed_count': len(obs), 'first_review_dbc': first_review_dbc,
        'observed_rate': len(obs)/obs_window_days,
        'rate_last_day': int(rate_last_day), 'rate_first_day': int(rate_first_day),
        'top_critic_frac': stats['top_critic_frac'], 'pub_diversity': stats['pub_diversity'],
        'pub_entropy': stats['pub_entropy'], 'low_activity_frac': stats['low_activity_frac'],
        'actual': actual,
    }

def wkde_predict(td):
    s = td['slug']
    scores = combined_score_with_scores(
        s, td['target_gap'], td['state']['observed_critics'], td['obs_window_days'],
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None
    try:
        profiles = build_weighted_critic_profiles(reviews_noon, close_date_map, scores, verbose=False)
        if len(profiles.df) == 0:
            return None
        model = build_weighted_kde_lambda_model(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window_custom(
            model, dbc_from=td['snap_dbc_eff'], dbc_to=td['midnight_utc_dbc'],
            observed_critics=td['state']['observed_critics'],
            observed_count=td['state']['observed_count'],
            first_review_dbc=td['state']['first_review_dbc'],
        )
        return float(pred)
    except Exception:
        return None

if CACHE.exists():
    df = pd.read_pickle(CACHE)
    print(f'Loaded {len(df)} cached rows')
else:
    rows = []
    for slug in close_date_map:
        td = target_data(slug)
        if td is None:
            continue
        row = {k: v for k, v in td.items() if k not in ('state', 'snap_time')}
        row['wkde_pred'] = wkde_predict(td)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
df = df.dropna(subset=FEATURES + ['actual', 'wkde_pred']).reset_index(drop=True)
print(f'Usable rows: {len(df)}')

## Ridge predictions via cohort 5-fold CV + h/m holdout

In [ ]:
cohort = df[~df['slug'].isin(HM)].reset_index(drop=True)
hm = df[df['slug'].isin(HM)].reset_index(drop=True)

X_c = cohort[FEATURES].values
y_c = cohort['actual'].values
X_hm = hm[FEATURES].values

# 5-fold CV on cohort
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cohort_ridge = np.zeros(len(cohort))
for train_idx, test_idx in kf.split(X_c):
    m = Ridge(alpha=10.0)
    m.fit(X_c[train_idx], y_c[train_idx])
    cohort_ridge[test_idx] = m.predict(X_c[test_idx])

# H/m: train on all cohort
ridge_all = Ridge(alpha=10.0)
ridge_all.fit(X_c, y_c)
hm_ridge = ridge_all.predict(X_hm)

ridge_map = dict(zip(cohort['slug'], cohort_ridge))
for s, p in zip(hm['slug'], hm_ridge):
    ridge_map[s] = p
df['ridge_pred'] = df['slug'].map(ridge_map)
print('Ridge predictions computed')

## Blend sweep — constant w

In [ ]:
df['err_wkde'] = df['wkde_pred'] - df['actual']
df['err_ridge'] = df['ridge_pred'] - df['actual']

def summarize_blend(subset, label, weights):
    print(f'\n{label} (n={len(subset)}):')
    print(f'  {"w_ridge":>10s}  {"MAE":>8s}  {"mean_err":>10s}')
    # Also component-wise
    for w in weights:
        pred = w * subset['ridge_pred'] + (1-w) * subset['wkde_pred']
        err = pred - subset['actual']
        mae = err.abs().mean()
        me = err.mean()
        print(f'  {w:>10.2f}  {mae:>8.2f}  {me:>+10.2f}')

weights = [0.0, 0.25, 0.5, 0.75, 1.0]
cohort_df = df[~df['slug'].isin(HM)]
hm_df = df[df['slug'].isin(HM)]

summarize_blend(cohort_df, 'Cohort', weights)
summarize_blend(hm_df, 'H/m', weights)

## Adaptive blend: w = min(1, n_obs/threshold)

If Ridge leans on recent-observation signals (rate_last_day is dominant feature), it should help more when we have more observed data. Test thresholds.

In [ ]:
print('Adaptive blend: w_ridge = min(1, n_obs/threshold)\n')
print(f'  {"threshold":>12s}  {"cohort_MAE":>12s}  {"cohort_me":>10s}  {"h/m_MAE":>8s}  {"h/m_me":>8s}')
for threshold in [40, 80, 120, 160, 200]:
    df['w_ridge'] = np.minimum(1.0, df['observed_count'] / threshold)
    df['pred_blend'] = df['w_ridge'] * df['ridge_pred'] + (1 - df['w_ridge']) * df['wkde_pred']
    df['err_blend'] = df['pred_blend'] - df['actual']
    cohort_e = df[~df['slug'].isin(HM)]['err_blend']
    hm_e = df[df['slug'].isin(HM)]['err_blend']
    print(f'  {threshold:>12d}  {cohort_e.abs().mean():>12.2f}  {cohort_e.mean():>+10.2f}  {hm_e.abs().mean():>8.2f}  {hm_e.mean():>+8.2f}')

## Per-target h/m comparison at best blend

In [ ]:
best_w = 0.5  # representative blend; adjust based on sweep above
df['pred_blend'] = best_w * df['ridge_pred'] + (1 - best_w) * df['wkde_pred']
df['err_blend'] = df['pred_blend'] - df['actual']

hm_df = df[df['slug'].isin(HM)].copy()
cols = ['slug', 'actual', 'wkde_pred', 'ridge_pred', 'pred_blend', 'err_wkde', 'err_ridge', 'err_blend']
print(f'Per-target h/m at w_ridge={best_w}:\n')
print(hm_df[cols].to_string(index=False, float_format='%.2f'))

## Decision

- **Blend dominates both** across cohort + h/m MAE AND calibration → ship as new primary.
- **One blend weight dominates for cohort, different one for h/m** → need adaptive blending (use observed_count or another signal).
- **Blend helps cohort but not h/m (or vice versa)** → limited synergy; ship individual.